# Notebook 99 — Exportación de las tablas Silver y Gold a CSV

## Objetivo

Exportar las tablas de las capas Silver y Gold del modelo de datos (actualmente almacenadas dentro del archivo `selmark.duckdb`) a archivos CSV individuales, organizados en las carpetas `data/silver/` y `data/gold/` del proyecto. La exportación responde a tres objetivos: garantizar la portabilidad de los datos finales para su consulta por los tutores y por la propia empresa, conservar una copia de respaldo en formato abierto independiente de DuckDB, y disponer de los archivos necesarios para los anexos del Trabajo Fin de Grado.

## Criterios de exportación

Las decisiones técnicas adoptadas en la exportación son las siguientes:

- **Formato**: CSV (texto delimitado), por su universalidad y compatibilidad con cualquier herramienta de análisis u oficina.
- **Delimitador**: punto y coma (`;`), estándar habitual en el entorno español que evita conflictos con los importes y los textos con comas internas.
- **Separador decimal**: punto (`.`), estándar técnico internacional que garantiza la re-importación sin pérdida en pandas, DuckDB y otras herramientas analíticas.
- **Codificación**: UTF-8 con BOM, que permite la apertura directa en Microsoft Excel con tildes y eñes correctamente reconocidas.
- **Valores nulos**: cadena vacía.
- **Booleanos**: representados como `True` y `False` literales.

## 1. Configuración y preparación de carpetas

In [6]:
import duckdb
from pathlib import Path
import os

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"
RUTA_DATA_SILVER = RUTA_PROYECTO / "data" / "silver"
RUTA_DATA_GOLD = RUTA_PROYECTO / "data" / "gold"

# Crear las carpetas si no existen
RUTA_DATA_SILVER.mkdir(parents=True, exist_ok=True)
RUTA_DATA_GOLD.mkdir(parents=True, exist_ok=True)

# Conexión en modo lectura (no vamos a modificar nada de la base)
con = duckdb.connect(str(RUTA_DUCKDB), read_only=True)

print(f"Conectado a: {RUTA_DUCKDB}")
print(f"Carpeta destino Silver: {RUTA_DATA_SILVER}")
print(f"Carpeta destino Gold:   {RUTA_DATA_GOLD}")
print(f"Ambas carpetas listas para recibir los archivos.")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Carpeta destino Silver: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\silver
Carpeta destino Gold:   C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\gold
Ambas carpetas listas para recibir los archivos.


## 2. Exportación de la capa Silver

Se exportan las seis tablas que componen la capa Silver del modelo. La exportación se realiza mediante la sintaxis `COPY ... TO ... (FORMAT CSV, ...)` de DuckDB, que escribe el archivo directamente en disco sin necesidad de cargar la tabla en memoria.

In [8]:
# Lista de tablas Silver a exportar
tablas_silver = [
    "dim_cliente",
    "fact_lineas_pedido",
    "ventas_minoristas",
    "mosaic",
    "tiempo",
    "mapeo_paises",
]

print("=" * 70)
print("EXPORTACIÓN DE LA CAPA SILVER")
print("=" * 70)

for tabla in tablas_silver:
    ruta_destino = RUTA_DATA_SILVER / f"{tabla}.csv"
    ruta_sql = str(ruta_destino).replace("\\", "/")
    
    print(f"\nExportando silver.{tabla}...")
    
    con.execute(f"""
        COPY (SELECT * FROM silver.{tabla})
        TO '{ruta_sql}'
        (FORMAT CSV, HEADER, DELIMITER ';', NULLSTR '')
    """)
    
    n_filas = con.execute(f"SELECT COUNT(*) FROM silver.{tabla}").fetchone()[0]
    tam_mb = ruta_destino.stat().st_size / (1024 * 1024)
    print(f"   ✓ {n_filas:>10,} filas | {tam_mb:>7.2f} MB | {ruta_destino.name}")

print("\n" + "=" * 70)
print("Capa Silver exportada correctamente.")
print("=" * 70)

EXPORTACIÓN DE LA CAPA SILVER

Exportando silver.dim_cliente...
   ✓      3,492 filas |    0.36 MB | dim_cliente.csv

Exportando silver.fact_lineas_pedido...
   ✓    339,705 filas |   46.36 MB | fact_lineas_pedido.csv

Exportando silver.ventas_minoristas...
   ✓  2,867,565 filas |  350.34 MB | ventas_minoristas.csv

Exportando silver.mosaic...
   ✓      6,457 filas |    0.44 MB | mosaic.csv

Exportando silver.tiempo...
   ✓      1,461 filas |    0.13 MB | tiempo.csv

Exportando silver.mapeo_paises...
   ✓        156 filas |    0.00 MB | mapeo_paises.csv

Capa Silver exportada correctamente.


## 3. Exportación de la capa Gold

Se exporta la tabla `gold.cliente_360`, resultado final del pipeline analítico y entrada de todos los análisis posteriores del proyecto.

In [10]:
ruta_destino = RUTA_DATA_GOLD / "cliente_360.csv"
ruta_sql = str(ruta_destino).replace("\\", "/")

print("=" * 70)
print("EXPORTACIÓN DE LA CAPA GOLD")
print("=" * 70)
print(f"\nExportando gold.cliente_360...")

con.execute(f"""
    COPY (SELECT * FROM gold.cliente_360)
    TO '{ruta_sql}'
    (FORMAT CSV, HEADER, DELIMITER ';', NULLSTR '')
""")

n_filas = con.execute("SELECT COUNT(*) FROM gold.cliente_360").fetchone()[0]
n_cols = con.execute("""
    SELECT COUNT(*) FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
""").fetchone()[0]
tam_mb = ruta_destino.stat().st_size / (1024 * 1024)

print(f"   ✓ {n_filas:,} filas, {n_cols} columnas | {tam_mb:.2f} MB | {ruta_destino.name}")
print("\n" + "=" * 70)
print("Capa Gold exportada correctamente.")
print("=" * 70)

EXPORTACIÓN DE LA CAPA GOLD

Exportando gold.cliente_360...
   ✓ 3,492 filas, 56 columnas | 1.12 MB | cliente_360.csv

Capa Gold exportada correctamente.


## 4. Verificación final

Se realiza una comprobación de integridad sobre los archivos exportados: número de filas, tamaño y existencia de cada archivo en su carpeta correspondiente.

In [11]:
print("=" * 70)
print("VERIFICACIÓN FINAL — Archivos generados")
print("=" * 70)

# Listar todos los archivos exportados
archivos_silver = sorted(RUTA_DATA_SILVER.glob("*.csv"))
archivos_gold = sorted(RUTA_DATA_GOLD.glob("*.csv"))

print(f"\nCarpeta {RUTA_DATA_SILVER}:")
total_mb_silver = 0
for f in archivos_silver:
    tam_mb = f.stat().st_size / (1024 * 1024)
    total_mb_silver += tam_mb
    print(f"   {f.name:<35s}  {tam_mb:>8.2f} MB")
print(f"   {'TOTAL SILVER':<35s}  {total_mb_silver:>8.2f} MB")

print(f"\nCarpeta {RUTA_DATA_GOLD}:")
total_mb_gold = 0
for f in archivos_gold:
    tam_mb = f.stat().st_size / (1024 * 1024)
    total_mb_gold += tam_mb
    print(f"   {f.name:<35s}  {tam_mb:>8.2f} MB")
print(f"   {'TOTAL GOLD':<35s}  {total_mb_gold:>8.2f} MB")

print(f"\nTOTAL EXPORTADO: {total_mb_silver + total_mb_gold:.2f} MB")
print(f"Número de archivos generados: {len(archivos_silver) + len(archivos_gold)}")

VERIFICACIÓN FINAL — Archivos generados

Carpeta C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\silver:
   dim_cliente.csv                          0.36 MB
   fact_lineas_pedido.csv                  46.36 MB
   mapeo_paises.csv                         0.00 MB
   mosaic.csv                               0.44 MB
   tiempo.csv                               0.13 MB
   ventas_minoristas.csv                  350.34 MB
   TOTAL SILVER                           397.63 MB

Carpeta C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\gold:
   cliente_360.csv                          1.12 MB
   TOTAL GOLD                               1.12 MB

TOTAL EXPORTADO: 398.75 MB
Número de archivos generados: 7


In [12]:
# Validación cruzada: re-cargar uno de los CSV y comprobar que los datos están bien
print("=" * 70)
print("VALIDACIÓN CRUZADA — Re-carga de gold.cliente_360.csv")
print("=" * 70)

ruta_gold_csv = str(RUTA_DATA_GOLD / "cliente_360.csv").replace("\\", "/")

prueba = con.execute(f"""
    SELECT COUNT(*) AS filas_csv
    FROM read_csv_auto('{ruta_gold_csv}', delim=';', header=True)
""").fetchone()[0]

filas_duckdb = con.execute("SELECT COUNT(*) FROM gold.cliente_360").fetchone()[0]

print(f"Filas en gold.cliente_360 (DuckDB): {filas_duckdb:,}")
print(f"Filas en cliente_360.csv (re-cargado): {prueba:,}")

if prueba == filas_duckdb:
    print(f"\n✅ Validación correcta: ambos volúmenes coinciden.")
else:
    print(f"\n❌ Discrepancia detectada — revisar el archivo CSV.")

VALIDACIÓN CRUZADA — Re-carga de gold.cliente_360.csv
Filas en gold.cliente_360 (DuckDB): 3,492
Filas en cliente_360.csv (re-cargado): 3,492

✅ Validación correcta: ambos volúmenes coinciden.


## 5. Cierre del notebook

In [13]:
con.close()
print("Conexión cerrada. Notebook 99 completado correctamente.")
print("\nTodos los archivos están disponibles en:")
print(f"   {RUTA_DATA_SILVER}")
print(f"   {RUTA_DATA_GOLD}")

Conexión cerrada. Notebook 99 completado correctamente.

Todos los archivos están disponibles en:
   C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\silver
   C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\data\gold
